In [0]:
# 1. Dados simulados de vendas de e-commerce (Camada Bronze)
dados_vendas = [
    ("PED-001", "Notebook Dell", "Eletrônicos", 3500.0, "2026-01-15", "SP"),
    ("PED-002", "Mouse Sem Fio", "Acessórios", 80.0, "2026-01-16", "RJ"),
    ("PED-003", "Teclado Mecânico", "Acessórios", 250.0, "2026-01-17", "MG"),
    ("PED-004", "Monitor 27", "Eletrônicos", 1200.0, "2026-01-18", "SP"),
    ("PED-005", "Cadeira Gamer", "Móveis", 950.0, "2026-01-19", "PR"),
    ("PED-006", "Mouse Sem Fio", "Acessórios", 80.0, "2026-01-20", "RJ"), # Registro para tratarmos depois
]

colunas = ["id_pedido", "produto", "categoria", "valor", "data_pedido", "estado"]

# 2. Criando o DataFrame distribuído com Spark
df_bronze = spark.createDataFrame(dados_vendas, colunas)

# 3. Exibindo os dados na tela com a ferramenta visual do Databricks
display(df_bronze)

id_pedido,produto,categoria,valor,data_pedido,estado
PED-001,Notebook Dell,Eletrônicos,3500.0,2026-01-15,SP
PED-002,Mouse Sem Fio,Acessórios,80.0,2026-01-16,RJ
PED-003,Teclado Mecânico,Acessórios,250.0,2026-01-17,MG
PED-004,Monitor 27,Eletrônicos,1200.0,2026-01-18,SP
PED-005,Cadeira Gamer,Móveis,950.0,2026-01-19,PR
PED-006,Mouse Sem Fio,Acessórios,80.0,2026-01-20,RJ


In [0]:
from pyspark.sql.functions import col, to_date, upper

# 1. Transformando os dados brutos na Camada Silver
df_silver = (
    df_bronze
    # Converte data_pedido de texto para o tipo Data de verdade
    .withColumn("data_pedido", to_date(col("data_pedido"), "yyyy-MM-dd"))
    # Padroniza a categoria com letras maiúsculas
    .withColumn("categoria", upper(col("categoria")))
    # Cria uma nova coluna calculada: imposto estimado de 10%
    .withColumn("imposto", col("valor") * 0.10)
)

# 2. Exibindo os dados limpos e transformados
display(df_silver)

id_pedido,produto,categoria,valor,data_pedido,estado,imposto
PED-001,Notebook Dell,ELETRÔNICOS,3500.0,2026-01-15,SP,350.0
PED-002,Mouse Sem Fio,ACESSÓRIOS,80.0,2026-01-16,RJ,8.0
PED-003,Teclado Mecânico,ACESSÓRIOS,250.0,2026-01-17,MG,25.0
PED-004,Monitor 27,ELETRÔNICOS,1200.0,2026-01-18,SP,120.0
PED-005,Cadeira Gamer,MÓVEIS,950.0,2026-01-19,PR,95.0
PED-006,Mouse Sem Fio,ACESSÓRIOS,80.0,2026-01-20,RJ,8.0


In [0]:
from pyspark.sql.functions import sum as _sum, count, round

# 1. Agrupando os dados por Categoria para gerar métricas de negócio (Camada Gold)
df_gold_categoria = (
    df_silver
    .groupBy("categoria")
    .agg(
        round(_sum("valor"), 2).alias("faturamento_total"),
        round(_sum("imposto"), 2).alias("imposto_total"),
        count("id_pedido").alias("total_pedidos")
    )
    .orderBy(col("faturamento_total").desc())
)

# 2. Exibindo o resumo de negócio
display(df_gold_categoria)

categoria,faturamento_total,imposto_total,total_pedidos
ELETRÔNICOS,4700.0,470.0,2
MÓVEIS,950.0,95.0,1
ACESSÓRIOS,410.0,41.0,3


Databricks visualization. Run in Databricks to view.